# Inference API Demo
This notebook demonstrates the backend-based MorphCell inference API.
We will cover:
1. **Model Initialization**
2. **Capability Inspection**
3. **Feature Extraction with `FeatureBundle`**
4. **Reconstruction**
   - Reconstruction from Input
   - Reconstruction from Features
   - Fusion Reconstruction
5. **MorphCell Features**
   - Cross-Reconstruction
   - Saliency Analysis
6. **Baseline Model Usage**

The public API now uses neutral method names such as:
- `extract_features`
- `extract_global_features`
- `extract_local_features`
- `reconstruct`
- `reconstruct_from_features`
- `fusion_reconstruct`

Returned features are stored in a `FeatureBundle`:
- `global_features`
- `local_features`
- `concated_features`
- `aux`

## 1. Setup 

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: /home/verve/Project/research/MorphCell


In [2]:
import torch
from morphcell.data import HDF5Dataset
from morphcell import MorphCellInference
from morphcell.utils.io import plot_ply

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.7.1+cu118
CUDA available: True
Using device: cuda


## 2. Model & Data Initialization


In [3]:
# Initialize model
inferencer = MorphCellInference(
    config_dir=f"{project_root}/morphcell/config",
    config_name="system/morphcell_cross+self",
    checkpoint_path=f"{project_root}/checkpoints/morphcell_cross_self.ckpt",
    device="auto",
)
print(f"Model successfully loaded on: {inferencer.get_device()}")
print(f"Capabilities: {inferencer.get_capabilities()}")

# Initialize Dataset
dataset = HDF5Dataset(
    path=f"{project_root}/datasets/redblood",
    splits=["set1"],
    normalize=True,
    normalize_scale=None,
    pca_align=True,
)

print(f"Dataset loaded with {len(dataset)} samples")

/home/verve/Project/research/MorphCell/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model successfully loaded on: cuda
Capabilities: ModelCapabilities(global_features=True, local_features=True, reconstruct_from_input=True, reconstruct_from_features=True, feature_fusion=True, cross_reconstruction=True, saliency=True)
Dataset loaded with 620 samples


## 3. Feature Extraction
Extract features from a point cloud into a `FeatureBundle`.
Input can be a `numpy.ndarray` or `torch.Tensor` of shape `(N, 3)` or `(B, N, 3)`.

For MorphCell:
- `global_features` corresponds to the global token representation
- `local_features` corresponds to patch features
- `concated_features` is the concatenated global + max-pooled local representation
- `aux` stores backend-specific extra tensors such as centers and groups

For baseline models:
- `global_features` is the global feature vector
- `local_features` is `None`
- `concated_features` is the same as `global_features`

In [4]:
# Get a sample from the dataset
sample_idx = 606
sample_data = dataset[sample_idx]
input_pc = sample_data["points"]
print(f"Input Point Cloud Shape: {input_pc.shape}, name: {sample_data['name']}")

# Extract features
features = inferencer.extract_features(input_pc)

print(type(features))
print("--- Feature Shapes ---")
print(f"global_features: {features.global_features.shape}")
print(
    f"local_features: {None if features.local_features is None else features.local_features.shape}"
)
print(
    f"concated_features: {None if features.concated_features is None else features.concated_features.shape}"
)
print(f"aux keys: {list(features.aux.keys())}")

# Convenience accessors
print("global only:", inferencer.extract_global_features(input_pc).shape)
print("local only:", inferencer.extract_local_features(input_pc).shape)

Input Point Cloud Shape: torch.Size([2048, 3]), name: knizocyte
<class 'morphcell.api.types.FeatureBundle'>
--- Feature Shapes ---
global_features: torch.Size([1, 384])
local_features: torch.Size([1, 64, 384])
concated_features: torch.Size([1, 768])
aux keys: ['centers', 'group']
global only: torch.Size([1, 384])
local only: torch.Size([1, 64, 384])


## 4. Reconstruction Methods
The unified API is `reconstruct`. You can reconstruct directly from input, or decode from a previously extracted `FeatureBundle`.

### 4.1 Reconstruction from Input
Reconstruct the input point cloud using the backend's reconstruction path.
Useful for verifying if the model captures the geometry.

In [5]:
# Perform reconstruction from input
reconstructed_pc = inferencer.reconstruct(input_pc)
print(f"Original shape: {input_pc.shape}")
print(f"Reconstructed shape: {reconstructed_pc.shape}")

plot_ply(input_pc, "Original")
plot_ply(reconstructed_pc, "Reconstructed")

Original shape: torch.Size([2048, 3])
Reconstructed shape: (2048, 3)


### 4.2 Reconstruction from Features
Extract features once, then decode the returned `FeatureBundle` directly. This is useful when you want to manipulate features or avoid recomputing them.

In [6]:
# Extract features once
features = inferencer.extract_features(input_pc)

# Decode directly from FeatureBundle
recon_from_features = inferencer.reconstruct_from_features(features)

print(f"Reconstruction from features: {recon_from_features.shape}")
plot_ply(recon_from_features, "Reconstructed from Features")

Reconstruction from features: (1, 2048, 3)


## 5. MorphCell-only Features
Some capabilities are backend-specific. MorphCell supports cross-reconstruction, while baseline models do not.

In [7]:
# Cross-reconstruction is available for MorphCell backends
cross_results = inferencer.cross_reconstruct(input_pc)

print("--- Cross Reconstruction Results ---")
for key, val in cross_results.items():
    if hasattr(val, "shape"):
        print(f"{key}: {val.shape}")

--- Cross Reconstruction Results ---
view1: (2022, 3)
view2: (1232, 3)
view1_rot: (2022, 3)
view2_rot: (1232, 3)
cross_recon1: (2048, 3)
cross_recon2: (2048, 3)
group1: (2048, 3)
group2: (2048, 3)


In [8]:
plot_ply(cross_results["group1"], "Group 1")
plot_ply(cross_results["group2"], "Group 2")
plot_ply(cross_results["cross_recon1"], "Cross Reconstruction 1")
plot_ply(cross_results["cross_recon2"], "Cross Reconstruction 2")

## 6. Baseline Model Usage
Baseline backends expose the same API, but only support global features and reconstruction-related methods.

In [9]:
baseline = MorphCellInference(
    config_dir=f"{project_root}/morphcell/config",
    config_name="system/dfn_dgcnn",
    checkpoint_path=f"{project_root}/checkpoints/dfn_dgcnn.ckpt",
    device="auto",
)

print(baseline.get_capabilities())
features = baseline.extract_features(input_pc)
print(features.global_features.shape)
print(features.local_features)  # None

recon = baseline.reconstruct(input_pc)
recon_from_features = baseline.reconstruct_from_features(features)

plot_ply(recon, "Baseline Reconstruction")

ModelCapabilities(global_features=True, local_features=False, reconstruct_from_input=True, reconstruct_from_features=True, feature_fusion=True, cross_reconstruction=False, saliency=False)
torch.Size([1, 512])
None
